In [1]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "true")

'true'

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# 프로젝트 루트를 기준으로 경로를 설정합니다.
file_path = str(PROJECT_ROOT / "data/raw/pdf/saving_tips/1.pdf")
loader = PyPDFLoader(file_path=file_path)
docs = loader.load()

print(f"불러온 문서 수: {len(docs)}")
print(docs[1].page_content)

불러온 문서 수: 9
- 2 -
금융꿀팁 200선-㉚제 목사회초년생을 위한 금융꿀팁 7가지
사 례
l(사례1) 사회초년생 김준성(31세)씨는 신년모임에서 친구들로부터 결혼준비, 주택마련, 노후준비 등을 위해 장래에 필요한 자금계획을 세우고 다양한 금융상품에 가입하고 있다는 이야기를 들음.   김준성씨도 올해부터는 계획을 세워서 장래에 필요한 자금을 모아야겠다고 다짐하였으나 막상 무엇부터 어떻게 해야 할지 막막해 하고 있음. l(사례2) 사회초년생 이한별(28세)씨는 취업후 결혼을 계획하고 결혼자금으로 5천만원의 대출이 필요하여 은행을 방문해 상담하였으나 신용등급이 낮아 대출이 곤란하다며 거절당함.    그동안 쉽고 편리하다는 이유로 신용카드 현금서비스를 자주 이용하였고 TV광고에 자주 나오는 저축은행과 대부업체의 대출을 무심코 이용하여 신용등급이 낮아져 은행대출이 거절된 것을 알고 신용관리를 제대로 하지 못한 것을 후회하고 있음.l(사례3) 사회초년생 서동수(30세)씨는 작년 첫 월급날 보험설계사인 선배의 권유로 종신보험, 변액CI보험 등 여러 개의 보험을 가입함.       최근 서씨는 보험료 부담도 되고 결혼자금도 필요하여 종신보험을 해지하려고 보험사에 문의했으나 해약환급금이 거의 없어 손해가 발생한다는 사실을 알게 되었음.


In [3]:
# 검증: 문서가 정상적으로 로드되었는지 확인
assert len(docs) > 0, "문서가 로드되지 않았습니다."
assert all(hasattr(d, 'page_content') for d in docs), "문서 객체 형식이 올바르지 않습니다."

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

# 여기를 split_documents로 바꿨어요!
split_docs = text_splitter.split_documents(docs)
print(f"청킹된 문서 수: {len(split_docs)}")

청킹된 문서 수: 9


In [5]:
# 검증: 청킹이 정상적으로 수행되었는지 확인
assert len(split_docs) >= len(docs), "청킹 후 문서 수가 줄어들 수 없습니다."
assert len(split_docs) > 0, "청킹된 문서가 없습니다."

In [6]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x1105b4050>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1110b8f50>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

print(f"인덱싱된 총 벡터 수: {vectorstore.index.ntotal}")

인덱싱된 총 벡터 수: 9


In [8]:
# 검증: 벡터스토어에 인덱싱된 문서 확인
assert vectorstore.index.ntotal > 0, "벡터스토어에 인덱싱된 데이터가 없습니다."

In [9]:
retriever = vectorstore.as_retriever()

In [10]:
query = "사회초년생을 위한 금융꿀팁 7가지"

# invoke() 메서드로 문서 검색
res = retriever.invoke(query)

print(f"검색 결과 수: {len(res)}")
for i in res:
    print(i.page_content[:100] + "...")
    print("=" * 100)

검색 결과 수: 4
- 3 -
제 목사회초년생을 위한 금융꿀팁 7가지
꿀 팁
☞ 사회초년생을 위한 금융꿀팁 7가지를 기억하고 실천해 보세요.
 ① 「파인」과 친해지기 학교를 졸업하고 사회생활을 시작하...
- 7 -
제 목사회초년생을 위한 금융꿀팁 7가지   ⑤ 종잣돈 모으기   사회생활을 시작하여 월급을 받게 되면 비록 적은 금액이라도 꾸준히 저축하여 이른바 “종잣돈”을 모으는 것...
- 9 -
제 목사회초년생을 위한 금융꿀팁 7가지  ⑦ 현금서비스 등 자제하기 사회생활을 시작하면 의외로 돈 쓸 곳이 많아져 현금서비스나 카드론을 이용하고, 심지어 대부업체에서 고...
- 2 -
금융꿀팁 200선-㉚제 목사회초년생을 위한 금융꿀팁 7가지
사 례
l(사례1) 사회초년생 김준성(31세)씨는 신년모임에서 친구들로부터 결혼준비, 주택마련, 노후준비 등을...


In [11]:
# 검증: 검색 결과가 존재하는지 확인
assert len(res) > 0, "검색 결과가 반환되지 않았습니다."